# Brain agent-type hot-swap (Scope A) — Design Spec

| Field | Value |
|---|---|
| **Date** | 2026-08-08 |
| **Status** | Draft for review — **NS-Mermaid all verified** (5/5 cells) |
| **User journey** | Working with `grok` brain → switch to `opencode` (or any brain-capable agent) without restarting `spur` |
| **Product surfaces** | `/brain [<name>]` · bare `/brain` picker · `/brains` listing · existing `BrainSpawned` session-detail rebuild |
| **Related designs** | `2026-07-05-multi-brain-design.md` (parent; Scope A subset) · `2026-04-14-session-switching-design.md` · agent model/effort pickers (orthogonal) |
| **Solve artifacts** | strategy `sol_2ef6bffa440b416f` · approach `sol_fb09bfe5515c46a8` · phase schedule max_pos=3 sat |

### One-line plan

Ship **multi-brain Scope A only**: mutable `active_brain_name` + `SwitchBrain` = retire → drop transport → spawn; rebuild `SessionDetail` via `BrainSpawned`; leave Scope B/C and in-place agent mutation alone.


In [ ]:
flowchart TD
    SPEC["`@spec BRAIN-SWITCH-STRATEGY
@type Strategy = enum[process_hotswap, in_place_mutate, concurrent_multi, restart_only]
@type Feasible = enum[ok, blocked]
@input strategy: Strategy
@input one_live_brain: Bool
@input agent_bound_to_subprocess: Bool
@input session_detail_agent_immutable: Bool
@output status: Feasible`"]

    HOTSWAP_OK["`@branch HOTSWAP_OK
@when strategy = process_hotswap and one_live_brain = true and agent_bound_to_subprocess = true and session_detail_agent_immutable = true
@ensures HOTSWAP_OK_STATUS: status = ok`"]

    HOTSWAP_BLOCK["`@branch HOTSWAP_BLOCK
@when strategy = process_hotswap and not (one_live_brain = true and agent_bound_to_subprocess = true and session_detail_agent_immutable = true)
@ensures HOTSWAP_BLOCK_STATUS: status = blocked`"]

    RESTART["`@branch RESTART
@when strategy = restart_only
@ensures RESTART_STATUS: status = ok`"]

    MUTATE["`@branch MUTATE
@when strategy = in_place_mutate
@ensures MUTATE_STATUS: status = blocked`"]

    CONCURRENT["`@branch CONCURRENT
@when strategy = concurrent_multi
@ensures CONCURRENT_STATUS: status = blocked`"]

    CHECK["`@verify DET: prove determinism
@verify COVER: prove partition_coverage
@verify EXCL: prove partition_exclusive
@verify STATUSES: witness each status
@verify HOTSWAP_OK_REACH: witness branch HOTSWAP_OK
@verify HOTSWAP_BLOCK_REACH: witness branch HOTSWAP_BLOCK
@verify RESTART_REACH: witness branch RESTART
@verify MUTATE_REACH: witness branch MUTATE
@verify CONCURRENT_REACH: witness branch CONCURRENT`"]

    SPEC --> HOTSWAP_OK --> CHECK
    SPEC --> HOTSWAP_BLOCK --> CHECK
    SPEC --> RESTART --> CHECK
    SPEC --> MUTATE --> CHECK
    SPEC --> CONCURRENT --> CHECK


## 1. Problem

Today SPUR pins the brain type at process start via `--brain <name>` and never mutates it:

- **One brain per process.** `run_interactive` holds `brain: Option<BrainSession>` — singular.
- **Brain type is fixed for the loop lifetime.** `brain_override: Option<String>` is threaded immutably into every `connect_brain` / `spawn_brain_session` call.
- **Session detail is agent-stamped at construction.** `SessionDetailView` keeps `agent_name` / `agent_cfg` as identity; `/clear` preserves them; mid-session knobs are `/model` and `/effort` only.

### 1.1 Consequences

1. Switching from `grok` to `opencode` requires shutting down `spur` and restarting with a different `--brain` flag.
2. Users cannot explore alternate brain backends in one interactive session.
3. Model/effort switching is **not** agent-type switching — same subprocess, different config option.

### 1.2 Hard architectural facts (immovable)

| Fact | Implication |
|---|---|
| Agent type bound to subprocess | Cannot hot-patch transport kind on a live connection |
| Session history is agent-specific | Cannot load a grok ACP session into opencode |
| SessionDetail agent identity immutable | View must be **rebuilt**, not field-mutated |
| Exactly one live brain | Concurrent multi-brain is Scope C (deferred) |


## 2. Goals and non-goals

### Goals (Scope A)

1. **In-process hot-swap** — `/brain opencode` retires the live grok brain and spawns a fresh opencode brain without leaving the TUI process.
2. **Discovery UX** — bare `/brain` opens a picker of `AgentRegistry::brain_capable()` agents; `/brains` is a read-only listing.
3. **Warm-restart semantics** — switch always retires; exactly one live brain; no background brains.
4. **Reuse lifecycle** — mechanically identical to existing `NewSession` (retire → spawn), with a different brain name and forced drop of stashed old-kind transport.
5. **Session detail correctness** — new view via existing `BrainSpawned` path with the new agent name/config.

### Non-goals

| Item | Why deferred |
|---|---|
| Scope B unified multi-agent session picker | Larger surface; not required for grok→opencode |
| Scope C concurrent live brains | Conflicts with one-live-brain |
| Conversation transplant across agents | Agent-specific stores; not sound |
| In-place `SessionDetailView.agent_name` mutation | Proven unsat vs architecture |
| Header agent-switch widget in session_detail core | Forces session_detail core touch; not needed for slash UX |
| Cross-restart persistence of last-used brain | Each terminal owns its brain selection |
| Gemini enum cleanup as hard prereq | Not load-bearing for hot-swap |
| New `BrainRegistry` type | Call `brain_capable()` directly in v1 |


## 3. Solver-backed design locks

Re-checked with `solve_constraints` (B′) and NS-Mermaid native proofs (this notebook).

### 3.1 B′ strategy / approach locks

| Lock | Status | Artifact | Implication |
|---|---|---|---|
| Strategy under hard arch | **sat** `process_hotswap` | `sol_2ef6bffa440b416f` | Only in-process path that works |
| `in_place_mutate` | **unsat** | assert-negation style | Do not mutate agent identity on view |
| `concurrent_multi` | **unsat** under one-live-brain | — | Scope C deferred |
| Ship model A_only + slash_and_picker | **sat** | `sol_fb09bfe5515c46a8` | Preferred frontier |
| Header widget without session_detail core | **unsat** | — | Use slash + picker only |
| Phase DAG max depth 3 with parallel foundation | **sat** | schedule solve | 3 ship phases |

### 3.2 NS-Mermaid proof cells (this notebook)

Executable specs below: strategy feasibility, switch dispatch, hot-swap sequence, session-detail action, ship scope.


## 4. Concepts

| Term | Definition |
|---|---|
| **Brain-capable agent** | Registry entry with role Brain or Both (`AgentRegistry::brain_capable`) |
| **Active brain name** | Mutable `String` in `run_interactive` replacing immutable `brain_override` |
| **Hot-swap** | Retire live brain + drop stashed transport + spawn target brain type |
| **Warm restart** | Fresh ACP session for the target agent; history not transplanted |
| **Model switch** | Mid-session `/model` / `/effort` on the **same** agent process (orthogonal) |
| **SessionDetail rebuild** | `BrainSpawned` constructs a new view with new `agent_name` + `agent_cfg` |


## 5. Runtime dispatch policy

`SwitchBrain { name: Option<String> }` outcomes are a total function of registry membership and active equality.


In [ ]:
flowchart TD
    SPEC["`@spec SWITCH-BRAIN-DISPATCH
@type SwitchOutcome = enum[noop, error, switched]
@input target_known: Bool
@input target_is_active: Bool
@output status: SwitchOutcome`"]

    ERROR["`@branch ERROR
@when target_known = false
@ensures ERROR_STATUS: status = error`"]

    NOOP["`@branch NOOP
@when target_known = true and target_is_active = true
@ensures NOOP_STATUS: status = noop`"]

    SWITCH["`@branch SWITCH
@when target_known = true and target_is_active = false
@ensures SWITCH_STATUS: status = switched`"]

    CHECK["`@verify DET: prove determinism
@verify COVER: prove partition_coverage
@verify EXCL: prove partition_exclusive
@verify STATUSES: witness each status
@verify ERROR_REACH: witness branch ERROR
@verify NOOP_REACH: witness branch NOOP
@verify SWITCH_REACH: witness branch SWITCH`"]

    SPEC --> ERROR --> CHECK
    SPEC --> NOOP --> CHECK
    SPEC --> SWITCH --> CHECK


### 5.1 Handler pseudocode

```text
on SwitchBrain { name: Some(target) }:
  if target not in registry.brain_capable().names:
      emit BrainSwitchError { name: target, available }
      return
  if target == active_brain_name:
      emit BrainSwitchNoop { name: target }
      return
  retire_active_brain(...)           // existing NewSession path
  drop(agent_connection.take())      // old kind transport not reusable
  active_brain_name = target
  spawn_brain_session(active_brain_name, permission_tx)
  emit BrainSwitched { from, to: target }
  // BrainSpawned / AgentSessionReady follow from spawn path

on SwitchBrain { name: None }:
  emit BrainPickerOpen { brains, active }

on ListBrains:
  emit BrainsListed { brains, active }
```

### 5.2 Key invariant

`retire → drop-stashed → spawn` maintains **exactly one live brain** at all times (except the brief teardown gap, identical to `NewSession`). No background brains, no transport pool.


## 6. Hot-swap sequence (grok → opencode)

Ordered protocol with alt branches for switched / noop / error.


In [ ]:
sequenceDiagram
    participant U as User
    participant TUI as SpurTUI
    participant LOOP as InteractiveLoop
    participant REG as AgentRegistry
    participant ORCH as Orchestrator

    Note over U,ORCH: @spec BRAIN-HOTSWAP-SEQUENCE<br/>@type RunStatus = enum[switched, noop, error]<br/>@input target_known: Bool<br/>@input target_is_active: Bool<br/>@output status: RunStatus

    U->>TUI: slash_brain
    Note over U,TUI: @message SLASH<br/>@from U<br/>@to TUI<br/>@event slash_brain<br/>@order 1<br/>@when true<br/>@ensures SLASH_SENT: true

    TUI->>LOOP: switch_brain
    Note over TUI,LOOP: @message SWITCH_CMD<br/>@from TUI<br/>@to LOOP<br/>@event switch_brain<br/>@order 2<br/>@when true<br/>@ensures CMD_SENT: true

    LOOP->>REG: lookup
    Note over LOOP,REG: @message LOOKUP<br/>@from LOOP<br/>@to REG<br/>@event lookup<br/>@order 3<br/>@when true<br/>@ensures LOOKUP_SENT: true

    alt target known and not active
        Note over LOOP,ORCH: @branch SWITCH<br/>@when target_known = true and target_is_active = false<br/>@ensures SWITCH_STATUS: status = switched
        LOOP->>ORCH: retire
        Note over LOOP,ORCH: @message RETIRE<br/>@from LOOP<br/>@to ORCH<br/>@event retire<br/>@order 4<br/>@when target_known = true and target_is_active = false<br/>@ensures RETIRE_SENT: true
        LOOP->>ORCH: spawn
        Note over LOOP,ORCH: @message SPAWN<br/>@from LOOP<br/>@to ORCH<br/>@event spawn<br/>@order 5<br/>@when target_known = true and target_is_active = false<br/>@ensures SPAWN_SENT: true
        ORCH-->>TUI: brain_switched
        Note over ORCH,TUI: @message SWITCHED_EVT<br/>@from ORCH<br/>@to TUI<br/>@event brain_switched<br/>@order 6<br/>@when target_known = true and target_is_active = false<br/>@ensures EVT_SENT: true
    else target known and active
        Note over LOOP,TUI: @branch NOOP<br/>@when target_known = true and target_is_active = true<br/>@ensures NOOP_STATUS: status = noop
        LOOP-->>TUI: brain_switch_noop
        Note over LOOP,TUI: @message NOOP_EVT<br/>@from LOOP<br/>@to TUI<br/>@event brain_switch_noop<br/>@order 7<br/>@when target_known = true and target_is_active = true<br/>@ensures NOOP_EVT_SENT: true
    else target unknown
        Note over LOOP,TUI: @branch ERROR<br/>@when target_known = false<br/>@ensures ERROR_STATUS: status = error
        LOOP-->>TUI: brain_switch_error
        Note over LOOP,TUI: @message ERROR_EVT<br/>@from LOOP<br/>@to TUI<br/>@event brain_switch_error<br/>@order 8<br/>@when target_known = false<br/>@ensures ERROR_EVT_SENT: true
    end

    Note over U,ORCH: @verify PROTO: prove sequence_protocol<br/>@verify DET: prove determinism<br/>@verify COVER: prove partition_coverage<br/>@verify EXCL: prove partition_exclusive<br/>@verify STATUSES: witness each status<br/>@verify SWITCH_REACH: witness branch SWITCH<br/>@verify NOOP_REACH: witness branch NOOP<br/>@verify ERROR_REACH: witness branch ERROR


## 7. Session detail policy

When the agent kind or session id changes, the view is **rebuilt**. Mutating `agent_name` in place is not a supported path for agent-type switch.


In [ ]:
flowchart TD
    SPEC["`@spec SESSION-DETAIL-ON-SWITCH
@type ViewAction = enum[rebuild, mutate_in_place]
@input session_id_changed: Bool
@input agent_kind_changed: Bool
@output status: ViewAction`"]

    REBUILD["`@branch REBUILD
@when session_id_changed = true or agent_kind_changed = true
@ensures REBUILD_ACTION: status = rebuild`"]

    MUTATE["`@branch MUTATE
@when session_id_changed = false and agent_kind_changed = false
@ensures MUTATE_ACTION: status = mutate_in_place`"]

    CHECK["`@verify DET: prove determinism
@verify COVER: prove partition_coverage
@verify EXCL: prove partition_exclusive
@verify STATUSES: witness each status
@verify REBUILD_REACH: witness branch REBUILD
@verify MUTATE_REACH: witness branch MUTATE`"]

    SPEC --> REBUILD --> CHECK
    SPEC --> MUTATE --> CHECK


### 7.1 Existing rebuild path (keep)

`App` already replaces `SessionDetailView` on `BrainSpawned` when the session id changes (`crates/spur-tui/src/app/events.rs`):

1. Resolve `agent_cfg` from registry by agent name.
2. `SessionDetailView::new_with_issue_snapshot(session, agent, "brain", cwd, agent_cfg, ...)`.
3. Navigate to `SessionDetail`.

**v1 rule:** do not add agent-mutation methods to `session_detail/`; optional status banner on `BrainSwitched` only.


## 8. Ship scope

MVP ships **Scope A only**. Scope B (unified picker) and Scope C (concurrent) are explicit defers.


In [ ]:
flowchart TD
    SPEC["`@spec SHIP-SCOPE-MVP
@type Scope = enum[A_only, A_plus_B, A_plus_B_plus_C]
@type Accept = enum[ship, defer]
@input scope: Scope
@output status: Accept`"]

    SHIP_A["`@branch SHIP_A
@when scope = A_only
@ensures SHIP_A_OK: status = ship`"]

    DEFER_B["`@branch DEFER_B
@when scope = A_plus_B
@ensures DEFER_B_OK: status = defer`"]

    DEFER_C["`@branch DEFER_C
@when scope = A_plus_B_plus_C
@ensures DEFER_C_OK: status = defer`"]

    CHECK["`@verify DET: prove determinism
@verify COVER: prove partition_coverage
@verify EXCL: prove partition_exclusive
@verify STATUSES: witness each status
@verify SHIP_A_REACH: witness branch SHIP_A
@verify DEFER_B_REACH: witness branch DEFER_B
@verify DEFER_C_REACH: witness branch DEFER_C`"]

    SPEC --> SHIP_A --> CHECK
    SPEC --> DEFER_B --> CHECK
    SPEC --> DEFER_C --> CHECK


## 9. Protocol surface

### 9.1 New `InteractiveInput` variants

```rust
InteractiveInput::SwitchBrain { name: Option<String> }, // /brain [<name>]
InteractiveInput::ListBrains,                            // /brains
```

### 9.2 New `SpurEventBody` variants (additive)

```rust
BrainSwitched { from: String, to: String },
BrainSwitchNoop { name: String },
BrainSwitchError { name: String, available: Vec<String> },
BrainsListed { brains: Vec<BrainInfo>, active: String },
BrainPickerOpen { brains: Vec<BrainInfo>, active: String },
```

`BrainInfo` may be a thin DTO `{ name, kind, is_default }` derived at emit time from `AgentConfig` — **no** persisted `BrainRegistry` type required for v1.

### 9.3 Round-trip tests

Per AGENTS.md: new event variants need tests in `crates/spur-acp/tests/executor_events_roundtrip.rs`.


## 10. Implementation phases (max depth 3)

Solver-feasible schedule with parallel foundation:

### Phase 1 — Foundations (parallel)

| Work | Crate |
|---|---|
| `InteractiveInput::SwitchBrain` / `ListBrains` | `spur-core` |
| `active_brain_name: String` replaces immutable `brain_override` | `interactive_loop` |
| New `SpurEventBody` variants + round-trip tests | `spur-acp` |

### Phase 2 — Switch handler

| Work | Crate |
|---|---|
| Validate via `brain_capable()`, noop/error/switch branches | `interactive_loop` |
| `retire → drop stashed transport → set name → spawn` | reuses `session.rs` lifecycle as-is |

### Phase 3 — Tests + TUI (parallel)

| Work | Crate |
|---|---|
| Orchestrator tests: switch / noop / error | `spur-core` |
| `/brain`, `/brains`, brain picker; handle events | `spur-tui` |
| Session detail: no agent mutation; optional status only | `session_detail` thin |

### File touch list (minimal)

| Touch | Skip for v1 |
|---|---|
| `spur-core/.../input.rs` | `SessionDetailView` agent identity fields |
| `spur-core/.../interactive_loop.rs` | Unified multi-agent session picker |
| `spur-acp/.../domain/events.rs` + roundtrips | Concurrent connection pool |
| `spur-tui` commands + brain picker | Conversation transplant |
| Existing `BrainSpawned` rebuild path | Gemini enum cleanup as gate |


## 11. Testing strategy

### Unit / orchestrator

- `switch_brain_retires_old_and_spawns_new_type`
- `switch_brain_unknown_name_emits_error_with_available_list`
- `switch_brain_same_name_emits_noop`
- `active_brain_name_used_by_subsequent_new_session`

### Serialization

- Round-trip all new `SpurEventBody` variants

### TUI

- Brain picker renders registry list with active marker
- `/brain opencode` dispatches `SwitchBrain`
- `BrainSpawned` after switch rebuilds session detail with new agent name

### Manual acceptance

```text
spur tui --brain grok
# work in session
/brain opencode
# expect BrainSwitched + new SessionDetail for opencode
/brains
# expect list with opencode active
/brain grok
# warm restart back
```


## 12. Risks and mitigations

| Risk | Mitigation |
|---|---|
| In-flight stream during switch | Cancel / turn-complete before retire (same discipline as `NewSession`) |
| Stashed transport kind mismatch | Always `drop(agent_connection.take())` on kind change |
| Delegations / MCP / notebook sockets leak | Rely on existing `retire_active_brain` completeness; add regression test |
| Users expect history transplant | Status copy: "switched brain — new session" |
| Spawn failure after retire | Same as `NewSession` failure path; emit error; user can retry `/brain` |


## 13. Relationship to multi-brain design

This notebook **implements Scope A** of `docs/superpowers/specs/2026-07-05-multi-brain-design.md` with these deliberate simplifications from the MCTS/solve pass:

1. **No `BrainRegistry` type** — call `AgentRegistry::brain_capable()` directly.
2. **No Gemini cleanup prereq** — not load-bearing for hot-swap.
3. **No Scope B in the same MVP** — unified picker remains a follow-up.
4. **Explicit NS-Mermaid proofs** for strategy, dispatch, sequence, view action, and ship scope.

Scope B follow-up (later notebook/PR): fan-out `ListSessions` + `SessionsListBatchComplete` + cross-brain `ResumeSession { brain_name }`.


## 14. Executable NS-Mermaid evidence

Diagrams authored against spur-notebook NS-Mermaid native profiles and executed via Notebook MCP:

| Spec cell | Profile | Obligations matched | `verified` |
|---|---|---:|---|
| `BRAIN-SWITCH-STRATEGY` | relational_lia / flowchart | 10/10 | true |
| `SWITCH-BRAIN-DISPATCH` | relational_lia / flowchart | 9/9 | true |
| `BRAIN-HOTSWAP-SEQUENCE` | sequence_trace / sequenceDiagram | 10/10 | true |
| `SESSION-DETAIL-ON-SWITCH` | relational_lia / flowchart | 7/7 | true |
| `SHIP-SCOPE-MVP` | relational_lia / flowchart | 8/8 | true |

All five cells report `solver_verified=true` and `proof_fresh=true` after run.

### Acceptance for this design document

- [x] All NS-Mermaid cells green
- [x] Aligns with solve artifacts listed in the header
- [x] Implementation plan matches 3-phase schedule
- [x] Explicit non-goals prevent Scope B/C creep
